In [73]:
%pip install torchtext torchdata

In [74]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import multi30k, Multi30k
from typing import Iterable, List
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torchdata.datapipes.iter import IterableWrapper, Mapper
import torchtext
from torchtext.vocab import build_vocab_from_iterator
from nltk.translate.bleu_score import sentence_bleu
import torch
import torch.nn as nn
import torch.optim as optim

import gradio as gr

import numpy as np
import random
import math
import time
from tqdm import tqdm
import matplotlib.pyplot as plt


# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [75]:
class Encoder(nn.Module):
    def __init__(self, vocab_len, emb_dim, hid_dim, n_layers, dropout_prob):
        super().__init__()
        self.hid_dim=hid_dim
        self.n_layers=n_layers
        self.embedding= nn.Embedding(vocab_len, emb_dim)
        self.lstm=nn.LSTM(emb_dim, hid_dim, n_layers, dropout = dropout_prob)
        self.dropout=nn.Dropout(dropout_prob)
    def forward(self,input_batch):
        embed=self.dropout(self.embedding(input_batch)) #transform the batch that contains the numero of words into embedding vectors and then apply the dropout to avoid the overfitting 
        embed=embed.to(device)
        outputs,(hidden,cell)=self.lstm(embed)
        return hidden ,cell 

In [76]:
#test du code :
vocab_len = 8
emb_dim = 10
hid_dim=8
n_layers=1
dropout_prob=0.5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder_t = Encoder(vocab_len, emb_dim, hid_dim, n_layers, dropout_prob).to(device)
print(encoder_t)

Encoder(
  (embedding): Embedding(8, 10)
  (lstm): LSTM(10, 8, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [77]:
src_batch = torch.tensor([[0,3,4,2,1]]) #the shape of this vector is [1,5] a sentence of 5 words but the Lstm prefers this input [ number of words ,number of sentences(length of batch)]
#  transpose the input tensor as the encoder LSTM is in Sequence_first mode by default
src_batch = src_batch.t().to(device)
print("Shape of input(src) tensor:", src_batch.shape)
hidden_t , cell_t = encoder_t(src_batch)
print("Hidden tensor from encoder:",hidden_t ,"\nCell tensor from encoder:", cell_t)

Shape of input(src) tensor: torch.Size([5, 1])
Hidden tensor from encoder: tensor([[[-0.3447,  0.0105,  0.0302, -0.1002, -0.5443,  0.3304, -0.1075,
           0.0613]]], grad_fn=<StackBackward0>) 
Cell tensor from encoder: tensor([[[-0.7070,  0.0239,  0.0604, -0.2555, -1.2618,  1.0701, -0.2192,
           0.1008]]], grad_fn=<StackBackward0>)


In [78]:
class Decoder(nn.Module):
    def __init__(self,output_dim,emb_dim,hid_dim,n_layers,dropout):
        super().__init__() 
        self.hid_dim=hid_dim 
        self.output_dim=output_dim 
        self.embedding=nn.Embedding(output_dim,emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout = dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.softmax = nn.LogSoftmax(dim=1)
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers
    def forward(self,input,hidden,cell):
        input = input.unsqueeze(0) # the LSTM still expects a "Sequence Length" at the start so the unsequeeze add 1 
        #before :[batch_size]---- after :input = [1, batch size]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]
        embedded = self.dropout(self.embedding(input)) #embedded = [1, batch size, emb dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction_logit = self.fc_out(output.squeeze(0))
        prediction = self.softmax(prediction_logit)
        #prediction = [batch size, output dim]
        #batch size number of different sentences
        return prediction, hidden, cell

        

In [ ]:
#decoder test 
output_dim = 6
emb_dim=256
hid_dim = 8
n_layers=1
dropout=0.5
decoder_t = Decoder(output_dim, emb_dim, hid_dim, n_layers, dropout).to(device)

In [80]:
input_t = torch.tensor([0]).to(device) #<bos>
input_t.shape
prediction, hidden, cell = decoder_t(input_t, hidden_t , cell_t)
print("Prediction:", prediction, '\nHidden:',hidden,'\nCell:', cell)

Prediction: tensor([[-1.5966, -1.7583, -1.9004, -1.8699, -2.0171, -1.6692]],
       grad_fn=<LogSoftmaxBackward0>) 
Hidden: tensor([[[-0.2358, -0.1311,  0.1230,  0.0397, -0.4420,  0.4606,  0.1870,
          -0.0357]]], grad_fn=<StackBackward0>) 
Cell: tensor([[[-0.4167, -0.2546,  0.4089,  0.3009, -0.9147,  1.1123,  0.5089,
          -0.0829]]], grad_fn=<StackBackward0>)


In [81]:
#Encoder-Decoder Connection manually :
#teacher_forcing_ratio is probability to use teacher forcing 
teacher_forcing_ratio = 0.5
trg = torch.tensor([[0],[2],[3],[5],[1]]).to(device) # 0 means start 1 means end , the tensore represents the target sentence 
#trg = [trg len, batch size]
batch_size = trg.shape[1]
trg_len = trg.shape[0]
trg_vocab_size = decoder_t.output_dim
#tensor to store decoder outputs
outputs_t = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device) #an output initially contains zeros 
#send to device
hidden_t = hidden_t.to(device)
cell_t = cell_t.to(device)
input = trg[0,:]
for t in range(1, trg_len):

    #you loop through the trg len and generate tokens
    #decoder receives previous generated token, cell and hidden
    # decoder outputs it prediction(probablity distribution for the next token) and updates hidden and cell
    output_t, hidden_t, cell_t = decoder_t(input, hidden_t, cell_t)

    #place predictions in a tensor holding predictions for each token
    outputs_t[t] = output_t

    # Select either the ground truth (teacher forcing) or the model's prediction as the next input based on the probability ratio.
    teacher_force = random.random() < teacher_forcing_ratio

    #get the highest predicted token from your predictions
    top1 = output_t.argmax(1)


    #if teacher forcing, use actual next token as next input
    #if not, use predicted token
    #input = trg[t] if teacher_force else top1
    input = trg[t] if teacher_force else top1

print(outputs_t,outputs_t.shape )


tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],

        [[-1.5430, -1.8036, -1.9697, -1.7929, -1.9824, -1.7268]],

        [[-1.5358, -1.9127, -1.9956, -1.6932, -1.9494, -1.7441]],

        [[-1.5575, -1.8587, -1.8286, -1.6836, -2.0801, -1.8189]],

        [[-1.6319, -1.8410, -1.7773, -1.6741, -2.1041, -1.7883]]],
       grad_fn=<CopySlices>) torch.Size([5, 1, 6])


In [82]:
#seq2seq class Encoder -> Context -> Decoder -> Loop
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device,trg_vocab):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.trg_vocab = trg_vocab

        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):
        #src = [src len, batch size]
        #trg = [trg len, batch size]
        #teacher_forcing_ratio is probability to use teacher forcing
        #e.g. if teacher_forcing_ratio is 0.75 you use ground-truth inputs 75% of the time


        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        #tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        #last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        hidden = hidden.to(device)
        cell = cell.to(device)


        #first input to the decoder is the <bos> tokens
        input = trg[0,:]

        for t in range(1, trg_len):

            #insert input token embedding, previous hidden and previous cell states
            #receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)

            #place predictions in a tensor holding predictions for each token
            outputs[t] = output

            #decide if you are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            #get the highest predicted token from your predictions
            top1 = output.argmax(1)


            #if teacher forcing, use actual next token as next input
            #if not, use predicted token
            #input = trg[t] if teacher_force else top1
            input = trg[t] if teacher_force else top1


        return outputs

In [83]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    # Wrap iterator with tqdm for progress logging
    train_iterator = tqdm(iterator, desc="Training", leave=False)

    for i, (src,trg) in enumerate(iterator):

        src = src.to(device)#src len number of words and batch size number of sentences 
        trg = trg.to(device)
        optimizer.zero_grad() #reset gradient each time otherwise they will be accumulated

        output = model(src, trg)

        #trg = [trg len, batch size]
        #output = [trg len, batch size, output dim]

        output_dim = output.shape[-1] #gives the vocabulary size 

        output = output[1:].view(-1, output_dim) #remoove first token which is usually <SOS> and we dont calculate loss on  it

        trg = trg[1:].contiguous().view(-1) #flatten into :[number of predictions ,vocab size ] because the cross entropy exects this

        #trg = [(trg len - 1) * batch size]
        #output = [(trg len - 1) * batch size, output dim]

        loss = criterion(output, trg) #CrossEntropy does:For each word:predicted probabilities compare correct word Then average loss.

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip) #very important because sequence models easily explode.

        optimizer.step() #Optimizer uses gradients to update weights 
        # Update tqdm progress bar with the current loss
        train_iterator.set_postfix(loss=loss.item())

        epoch_loss += loss.item()


    return epoch_loss / len(list(iterator))

In [84]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    # Wrap iterator with tqdm for progress logging
    valid_iterator = tqdm(iterator, desc="Training", leave=False)

    with torch.no_grad():

        for i, (src,trg) in enumerate(iterator):

            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, 0) #turn off teacher forcing
            # teacher forcing is turned off during evaluation to evaluate the model's ability to generate sequences based on its own predictions.

            #trg = [trg len, batch size]
            #output = [trg len, batch size, output dim]

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)

            trg = trg[1:].contiguous().view(-1)


            #trg = [(trg len - 1) * batch size]
            #output = [(trg len - 1) * batch size, output dim]

            loss = criterion(output, trg)
            # Update tqdm progress bar with the current loss
            valid_iterator.set_postfix(loss=loss.item())

            epoch_loss += loss.item()

    return epoch_loss / len(list(iterator))

In [85]:
import urllib.request
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Multi30K_de_en_dataloader.py'
urllib.request.urlretrieve(url, 'Multi30K_de_en_dataloader.py')
print("File downloaded successfully!")

File downloaded successfully!


In [86]:
# 1. Install spacy
%pip install spacy

# 2. Download the German language model 
!python -m spacy download de_core_news_sm

# 3. Download the English language model 
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
      --------------------------------------- 0.3/14.6 MB ? eta -:--:--
     - -------------------------------------- 0.5/14.6 MB 1.3 MB/s eta 0:00:11
     - -------------------------------------- 0.5/14.6 MB 1.3 MB/s eta 0:00:11
     -- ------------------------------------- 0.8/14.6 MB 1.0 MB/s eta 0:00:14
     -- ------------------------------------ 1.0/14.6 MB 882.6 kB/s eta 0:00:16
     --- ----------------------------------- 1.3/14.6 MB 972.7 kB/s eta 0:00:14
     ---- ----------------------------------- 1.6/14.6 MB 1.0 MB/s eta 0:00:13
     ----- ---------------------------------- 2.1/14.6 MB 1.2 MB/s eta 0:00:11
     ------- -------------------------------- 2.6/14.6 MB 1.4 MB/s eta 0:00:09
     ------- -------------------------------- 2.9/14.6 MB 1.4 MB/s eta 0:00:09
     --------- ------------------------------ 3.4/14.6 MB 1.4 MB/s eta 0:00:0


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 187, in _run_module_as_main
    mod_name, mod_spec, code = _get_module_details(mod_name, _Error)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 146, in _get_module_details
    return _get_module_details(pkg_main_name, error)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 110, in _get_module_details
    __import__(pkg_name)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\spacy\__init__.py", line 6, in <module>
    from .e

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 4.2 MB/s eta 0:00:03
     ---- ----------------------------------- 1.3/12.8 MB 3.5 MB/s eta 0:00:04
     ------- -------------------------------- 2.4/12.8 MB 4.1 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.8 MB 4.4 MB/s eta 0:00:03
     ------------ --------------------------- 3.9/12.8 MB 4.2 MB/s eta 0:00:03
     --------------- ------------------------ 5.0/12.8 MB 4.1 MB/s eta 0:00:02
     ------------------ --------------------- 5.8/12.8 MB 4.1 MB/s eta 0:00:02
     -------------------- ------------------- 6.6/12.8 MB 4.1 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 4.0 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 4.0 MB/s eta 0:00:02
     --------------------------- ------------ 8.9/12.8 MB 3.9 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 3


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 187, in _run_module_as_main
    mod_name, mod_spec, code = _get_module_details(mod_name, _Error)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 146, in _get_module_details
    return _get_module_details(pkg_main_name, error)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\runpy.py", line 110, in _get_module_details
    __import__(pkg_name)
  File "c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\spacy\__init__.py", line 6, in <module>
    from .e

In [87]:
%pip install portalocker>=2.0.0

Note: you may need to restart the kernel to use updated packages.


In [88]:
%run Multi30K_de_en_dataloader.py

In [89]:
train_dataloader, valid_dataloader = get_translation_dataloaders(batch_size = 4)#,flip=True)

In [90]:
src, trg = next(iter(train_dataloader))
src,trg

(tensor([[    2,     2,     2,     2],
         [    3,  5510,  5510, 12642],
         [    1,     3,     3,     8],
         [    1,     1,     1,  1701],
         [    1,     1,     1,     3]]),
 tensor([[   2,    2,    2,    2],
         [   3, 6650,  216,    6],
         [   1, 4623,  110, 3398],
         [   1,  259, 3913,  202],
         [   1,  172, 1650,  109],
         [   1, 9953, 3823,   37],
         [   1,  115,   71,    3],
         [   1,  692, 2808,    1],
         [   1, 3428, 2187,    1],
         [   1,    5,    5,    1],
         [   1,    3,    3,    1]]))

In [91]:
data_itr = iter(train_dataloader)
# sentences are sorted by length in the data 
# moving forward in the dataset to reach sequences of longer length for illustration purpose. (Remember the dataset is sorted on sequence len for optimal padding)
for n in range(1000):
    german, english= next(data_itr)

for n in range(3):
    german, english=next(data_itr)
    german=german.T
    english=english.T
    print("________________")
    print("german")
    for g in german:
        print(index_to_german(g)) #converts numbers back into human words 
    print("________________")
    print("english")
    for e in english:
        print(index_to_eng(e)) #converts numbers back into human words 


________________
german
<bos> Personen mit schwarzen Hüten in der Innenstadt . <eos>
<bos> Eine Gruppe Menschen protestiert in einer Stadt . <eos>
<bos> Eine Gruppe teilt ihre politischen Ansichten mit . <eos>
<bos> Mehrere Personen sitzen an einem felsigen Strand . <eos>
________________
english
<bos> People in black hats gathered together downtown . <eos> <pad> <pad> <pad>
<bos> A group of people protesting in a city . <eos> <pad> <pad>
<bos> A group is letting their political opinion be known . <eos> <pad>
<bos> A group of people are sitting on a rocky beach . <eos>
________________
german
<bos> Zwei sitzende Personen mit Hüten und Sonnenbrillen . <eos>
<bos> Ein kleiner Junge mit Hut beim Angeln . <eos>
<bos> Diese zwei Frauen haben Spaß im Giorgio's . <eos>
<bos> Zwei kleine Kinder schlafen auf dem Sofa . <eos>
________________
english
<bos> Two people sitting in hats and shades . <eos> <pad> <pad> <pad>
<bos> A young boy in a hat is fishing by himself . <eos>
<bos> These two wome

In [92]:
#This code sets the random seed for various libraries and modules in order to make the results reproducible:
import random
import numpy as np
import torch

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [93]:
INPUT_DIM = len(vocab_transform['de'])
OUTPUT_DIM = len(vocab_transform['en'])
ENC_EMB_DIM = 128 
DEC_EMB_DIM = 128 
HID_DIM = 256 
N_LAYERS = 1 
ENC_DROPOUT = 0.3 
DEC_DROPOUT = 0.3 
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device,trg_vocab = vocab_transform['en']).to(device)

In [94]:
#initializing the weights of the neural networks 
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(19214, 128)
    (lstm): LSTM(128, 256, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(10837, 128)
    (lstm): LSTM(128, 256, dropout=0.3)
    (fc_out): Linear(in_features=256, out_features=10837, bias=True)
    (softmax): LogSoftmax(dim=1)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (trg_vocab): Vocab()
)

In [95]:
# counts every single individual weight (number) that the model will be trying to learn and update during training.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 7,422,165 trainable parameters


In [96]:
optimizer = optim.Adam(model.parameters())

PAD_IDX = vocab_transform['en'].get_stoi()['<pad>']  #retrieves the index of the `<pad>` token in the target vocabulary.
criterion = nn.CrossEntropyLoss(ignore_index = PAD_IDX)

In [97]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [98]:
torch.cuda.empty_cache()

N_EPOCHS = 3 #run the training for at least 5 epochs
CLIP = 1

best_valid_loss = float('inf')
best_train_loss = float('inf')
train_losses = []
valid_losses = []

train_PPLs = []
valid_PPLs = []

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_dataloader, optimizer, criterion, CLIP)
    train_ppl = math.exp(train_loss)
    valid_loss = evaluate(model, valid_dataloader, criterion)
    valid_ppl = math.exp(valid_loss)


    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)


    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'RNN-TR-model.pt')

    train_losses.append(train_loss)
    train_PPLs.append(train_ppl)
    valid_losses.append(valid_loss)
    valid_PPLs.append(valid_ppl)

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {train_ppl:7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {valid_ppl:7.3f}')


KeyboardInterrupt: 

In [ ]:
#If you want to skip training and load the pre-trained model instead, run the following cell:
import urllib.request

# The URL 
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/RNN-TR-model.pt'
urllib.request.urlretrieve(url, 'RNN-TR-model.pt')

#load
model.load_state_dict(torch.load('RNN-TR-model.pt', map_location=torch.device('cpu')))

<All keys matched successfully>

In [100]:
import torch.nn.functional as F

def generate_translation(src_sentence, model=model, src_vocab=vocab_transform['de'], trg_vocab=vocab_transform['en'], max_len=50):
    model.eval()  # Set the model to evaluation mode

    with torch.no_grad():
        src_tensor = text_transform[SRC_LANGUAGE](src_sentence).view(-1, 1).to(device)

        # Pass the source tensor through the encoder
        hidden, cell = model.encoder(src_tensor)

        # Create a tensor to store the generated translation
        # get_stoi() maps tokens to indices
        trg_indexes = [trg_vocab.get_stoi()['<bos>']]  # Start with <bos> token

        # Convert the initial token to a PyTorch tensor
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(1)  # Add batch dimension

        # Move the tensor to the same device as the model
        trg_tensor = trg_tensor.to(model.device)


        # Generate the translation
        for _ in range(max_len):

            # Pass the target tensor and the previous hidden and cell states through the decoder
            output, hidden, cell = model.decoder(trg_tensor[-1], hidden, cell)

            # Get the predicted next token
            pred_token = output.argmax(1)[-1].item()

            # Append the predicted token to the translation
            trg_indexes.append(pred_token)


            # If the predicted token is the <eos> token, stop generating
            if pred_token == trg_vocab.get_stoi()['<eos>']:
                break

            # Convert the predicted token to a PyTorch tensor
            trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(1)  # Add batch dimension

            # Move the tensor to the same device as the model
            trg_tensor = trg_tensor.to(model.device)

        # Convert the generated tokens to text
        # get_itos() maps indices to tokens
        trg_tokens = [trg_vocab.get_itos()[i] for i in trg_indexes]

        # Remove the <sos> and <eos> from the translation
        if trg_tokens[0] == '<bos>':
            trg_tokens = trg_tokens[1:]
        if trg_tokens[-1] == '<eos>':
            trg_tokens = trg_tokens[:-1]

        # Return the translation list as a string

        translation = " ".join(trg_tokens)

        return translation

In [101]:
# model.load_state_dict(torch.load('RNN-TR-model.pt'))

# Actual translation: Asian man sweeping the walkway.
src_sentence = 'Ein asiatischer Mann kehrt den Gehweg.'


generated_translation = generate_translation(src_sentence=src_sentence, model=model, src_vocab=vocab_transform['de'], trg_vocab=vocab_transform['en'], max_len=12)
print(generated_translation)


An Asian man is on the sidewalk .


In [102]:
def calculate_bleu_score(generated_translation, reference_translations):
    # Convert the generated translations and reference translations into the expected format for sentence_bleu
    references = [reference.split() for reference in reference_translations]
    hypothesis = generated_translation.split()

    # Calculate the BLEU score
    bleu_score = sentence_bleu(references, hypothesis)

    return bleu_score

In [103]:
reference_translations = [
    "Asian man sweeping the walkway .",
    "An asian man sweeping the walkway .",
    "An Asian man sweeps the sidewalk .",
    "An Asian man is sweeping the sidewalk .",
    "An asian man is sweeping the walkway .",
    "Asian man sweeping the sidewalk ."
]

bleu_score = calculate_bleu_score(generated_translation, reference_translations)
print("BLEU Score:", bleu_score)

BLEU Score: 0.5


In [109]:
# Launch the interface
iface.launch(share=True,debug=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://225b2f8756f1982787.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://171ef47123063fb0da.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://225b2f8756f1982787.gradio.live
